# V126 - Heavyweight Kaggle build (T4 Commit)

Full-scale build for a Kaggle GPU Commit. It combines everything adopted: word-only chat, decoded assessment, mentions, and content views; per-module LightGBM-1200 plus Ridge heads; E5 dense; ten-bag ensembles with out-of-bag verdicts; OOF-weighted combination; and the consensus specialist action on the frozen champion with slice, safety, and geometry locks. Deliberately excluded with evidence: bigrams, stemming, CatBoostRanker (constant on thin signal), Qwen (loses, VRAM-hungry), blends without gates. SMOKE=1 verifies the whole path locally in minutes. Commit needs internet ON for pip and E5 weights, a T4 accelerator, the organizer dataset, and the champion file.

In [1]:
import json
import os
import re
import subprocess
import sys
from hashlib import sha256
from pathlib import Path

SMOKE = bool(int(os.getenv("V126_SMOKE", "0")))
try:
    import lightgbm, sentence_transformers  # noqa
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm==4.6.0", "sentence-transformers"])
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

SEED = 6161
N_SPLITS = 2 if SMOKE else 5
N_EST = 20 if SMOKE else 1200
BAGS = 2 if SMOKE else 10
GATE_FLOOR = 0.002
SLICE_FLOOR = 0.01
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

def find_file(name):
    roots = [Path("/kaggle/input"), Path(".")]
    hits = [p for r in roots if r.exists() for p in sorted(r.rglob(name))]
    return hits[0] if hits else None


def show_inputs():
    root = Path("/kaggle/input")
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file() and len(str(p)) < 160:
                print("input:", p)
    else:
        print("no /kaggle/input; local mode")


DATA_F = find_file("train_relevance.csv")
CHAMPION = find_file("submission_v42_primary.csv")
if DATA_F is None or CHAMPION is None:
    show_inputs()
assert DATA_F is not None, "train_relevance.csv not found: attach the organizer-data dataset to this notebook"
assert CHAMPION is not None, "submission_v42_primary.csv not found: attach the champion dataset to this notebook"
DATA = DATA_F.parent
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v126")
OUT.mkdir(parents=True, exist_ok=True)
print(f"SMOKE={SMOKE} DATA={DATA} lgb={lgb.__version__}")

SMOKE=True DATA=data lgb=4.7.0


## Views including E5

E5 encodes once and caches. Mentions count module keywords. Content is training-free syllabus cosine.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train, n_test = len(train), len(test_ids)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())
MN = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", (str(row["module_name"]) + " " + str(row["description_and_syllabus"])).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        MN[i, j] = sum(d.count(k) for k in keys)
MN = (MN - MN[:n_train].mean(0)) / (MN[:n_train].std(0) + 1e-9)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(docs + catalog["description_and_syllabus"].fillna("").tolist())
syll = [catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "" for m in MODS]
CM = cosine_similarity(vec.transform(docs), vec.transform(syll))
VIEWS = {"word": W, "assess": A, "mentions": np.nan_to_num(MN), "content": CM}
if not SMOKE:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("intfloat/multilingual-e5-base")
    E = np.asarray(enc.encode(["query: " + d for d in docs], batch_size=32, show_progress_bar=True, normalize_embeddings=True))
    VIEWS["e5"] = E
    print("views:", {k: v.shape for k, v in VIEWS.items()})

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)
is_nochat = np.array([u not in by_user for u in all_users])

def asdense(M, idx):
    m = M[idx] if not hasattr(M, "toarray") else M[idx].toarray()
    return np.asarray(m, dtype=np.float32)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

## Heavy heads, bagged base, consensus action, submission

Per-module LightGBM plus Ridge per view under grouped folds. Word view bags for the base verdict. The consensus specialist action from v115 composes onto the champion file behind its three locks. Fail-close writes the champion.

In [3]:
names = list(VIEWS)
oof = {k: np.zeros_like(Y) for k in names}
pred = {k: np.zeros((n_test, len(MODS))) for k in names}
fids = np.zeros(n_train, int)
outer = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(outer.split(asdense(VIEWS[names[0]], range(n_train)), strata, fold_groups)):
    for k in names:
        Xtr, Xva = asdense(VIEWS[k][:n_train], tr), asdense(VIEWS[k][:n_train], va)
        Xte = asdense(VIEWS[k][n_train:], range(n_test))
        for j in range(len(MODS)):
            r = Ridge(alpha=1.0)
            r.fit(Xtr, Y[tr, j])
            oof[k][va, j] = r.predict(Xva)
            pred[k][:, j] += r.predict(Xte) / N_SPLITS
            if k == "word":
                g = lgb.LGBMRegressor(objective="regression", n_estimators=N_EST, learning_rate=0.05,
                                        num_leaves=31, min_data_in_leaf=50, verbose=-1, deterministic=True, seed=SEED)
                g.fit(Xtr, Y[tr, j])
                oof[k][va, j] = (oof[k][va, j] + g.predict(Xva)) / 2
    fids[va] = fi + 1
    print(f"outer {fi + 1} done", flush=True)
for k in names:
    print(f"{k}: OOF={ndcg5(Y, oof[k]).mean():.5f}")
rb = np.random.RandomState(SEED)
acc = np.zeros_like(Y)
cnt = np.zeros(n_train)
for b in range(BAGS if not SMOKE else 2):
    idx = rb.choice(n_train, n_train, replace=True)
    oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(asdense(VIEWS["word"][:n_train], idx), Y[idx, j])
        acc[oob, j] += m.predict(asdense(VIEWS["word"][:n_train], oob))
    cnt[oob] += 1
oof_bag = acc / np.maximum(cnt[:, None], 1)
d = ndcg5(Y, oof_bag) - ndcg5(Y, oof["word"])
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"bagged-word OOF={ndcg5(Y, oof_bag).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}]")

from sklearn.model_selection import StratifiedKFold
SEEDS5 = [11, 22, 33, 44, 55]
nc_tr = np.where(is_nochat[:n_train])[0]
nc_te = np.where(is_nochat[n_train:])[0]
votes = np.zeros((len(nc_tr), len(MODS), len(SEEDS5)))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(nc_tr, Y[nc_tr].argmax(1)))
for si, s in enumerate(SEEDS5):
    rb = np.random.RandomState(s)
    for fi, (trs, vas) in enumerate(splits):
        acc5 = np.zeros((len(vas), len(MODS)))
        for b in range(BAGS if not SMOKE else 2):
            idx = nc_tr[trs][rb.choice(len(trs), len(trs), replace=True)]
            for j in range(len(MODS)):
                m = Ridge(alpha=1.0)
                m.fit(A[:n_train][idx], Y[idx, j])
                acc5[:, j] += m.predict(A[:n_train][nc_tr[vas]]) / (BAGS if not SMOKE else 2)
        votes[vas, :, si] = acc5
cons = np.zeros(len(nc_tr), int)
for i in range(len(nc_tr)):
    tops = votes[i].argmax(0)
    cons[i] = int((tops == np.bincount(tops).argmax()).sum())
print(f"consensus>=4: {(cons >= 4).sum()} / {len(nc_tr)}")
ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
report = {}
final = C.copy()
P = np.zeros((len(nc_te), len(MODS)))
T = np.zeros((len(nc_te), len(SEEDS5)), int)
for si, s in enumerate(SEEDS5):
    rb = np.random.RandomState(s)
    acc5 = np.zeros_like(P)
    for b in range(BAGS if not SMOKE else 2):
        idx = nc_tr[rb.choice(len(nc_tr), len(nc_tr), replace=True)]
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(A[:n_train][idx], Y[idx, j])
            acc5[:, j] += m.predict(A[n_train:][nc_te]) / (BAGS if not SMOKE else 2)
    P += acc5 / len(SEEDS5)
    T[:, si] = acc5.argmax(1)
agree = (T == np.apply_along_axis(lambda r: np.bincount(r).argmax(), 1, T)[:, None]).sum(1)
keep = nc_te[agree >= 4]
S = C.copy()
S[keep] = P[np.isin(nc_te, keep)]
changed = set(np.where((S != C).any(1))[0].tolist())
assert changed <= set(keep.tolist()), "geometry breach"
report.update({"changed": len(changed), "top1_changes": int((C.argmax(1) != S.argmax(1)).sum())})
print(f"consensus test users: {len(keep)} changed={len(changed)} top1-changes={report['top1_changes']}")
final = S
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v126_primary.csv", index=False)
report["sha"] = sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16]
report["champion_sha"] = sha256(pd.util.hash_pandas_object(ch).values.tobytes()).hexdigest()[:16]
Path(OUT / "v126_gate_report.json").write_text(json.dumps(report, indent=1))
print("wrote", OUT / "submission_v126_primary.csv", report)

outer 1 done


outer 2 done


word: OOF=0.62232
assess: OOF=0.43125
mentions: OOF=0.53125
content: OOF=0.41308


bagged-word OOF=0.49736 delta=-0.12496 CI95=[-0.13225,-0.11681]
consensus>=4: 57 / 117


/home/athilla/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


consensus test users: 20 changed=20 top1-changes=10
wrote results/v126/submission_v126_primary.csv {'changed': 20, 'top1_changes': 10, 'sha': '20a933fc6522dfbf', 'champion_sha': '7bc841d13df9f947'}
